# 卷积神经网络（上）卷积运算 + LeNet 手写数字识别

本节做三件事：

1. 从最朴素的 `for` 循环写一个二维卷积，对照 `nn.Conv2d` 输出验证一致。
2. 用手设计的 kernel 在真实图像上做**边缘检测**，直观感受卷积。
3. 实现 **LeNet-5** 在 MNIST 子集上做手写数字识别。

## 1. 二维卷积：手写 vs `nn.Conv2d`

信号处理里 "convolution" 严格意义上 kernel 要翻转，而深度学习里习惯说的 "convolution" 其实是**互相关（cross-correlation）**——kernel 不翻转，直接滑动相乘相加。`nn.Conv2d` 就是后者。

下面对单通道输入手写一遍：

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import matplotlib.pyplot as plt

torch.manual_seed(0)

def conv2d_naive(X, W, b=0.0, stride=1, padding=0):
    """X: [H, W], W: [kH, kW]; returns [H_out, W_out]. 仅做单通道、无 batch。"""
    if padding:
        X = F.pad(X, (padding, padding, padding, padding))
    H, Wd = X.shape; kH, kW = W.shape
    H_out = (H - kH) // stride + 1
    W_out = (Wd - kW) // stride + 1
    out = torch.zeros(H_out, W_out)
    for i in range(H_out):
        for j in range(W_out):
            h = i * stride; w = j * stride
            out[i, j] = (X[h:h+kH, w:w+kW] * W).sum() + b
    return out

x = torch.arange(25, dtype=torch.float32).reshape(5, 5)
k = torch.tensor([[1., 0., -1.], [1., 0., -1.], [1., 0., -1.]])
manual = conv2d_naive(x, k, stride=1, padding=0)
print('naive output:'); print(manual)

In [ ]:
# nn.Conv2d 的等价：[N, C_in, H, W] -> [N, C_out, H_out, W_out]
X = x.reshape(1, 1, 5, 5)
conv = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=0, bias=False)
with torch.no_grad():
    conv.weight.copy_(k.reshape(1, 1, 3, 3))
out_torch = conv(X).squeeze()
print('nn.Conv2d output:'); print(out_torch)
print('max abs diff:', (out_torch - manual).abs().max().item())

### 形状公式

$H_\text{out} = \lfloor (H + 2p - k) / s \rfloor + 1$，宽度同理。下面用 `nn.Conv2d` 直接验证 padding/stride 组合：

In [ ]:
X = torch.zeros(1, 1, 7, 7)
for k_size, pad, st in [(3, 0, 1), (3, 1, 1), (3, 1, 2), (5, 2, 1)]:
    c = nn.Conv2d(1, 1, kernel_size=k_size, padding=pad, stride=st)
    out = c(X)
    print(f'kernel={k_size} pad={pad} stride={st} -> {tuple(out.shape)}')

## 2. 边缘检测：手设计 kernel 在真实图像上

拉普拉斯算子 $\begin{bmatrix}0 & -1 & 0 \\ -1 & 4 & -1 \\ 0 & -1 & 0\end{bmatrix}$ 高亮像素与邻域的差异——即边缘。

In [ ]:
import os
from torchvision import datasets, transforms

CACHE = os.path.expanduser('~/.cache/torch_data')
mnist = datasets.MNIST(CACHE, train=True, download=True, transform=transforms.ToTensor())
img, label = mnist[0]      # [1, 28, 28]

laplacian = torch.tensor([[0., -1, 0], [-1, 4, -1], [0., -1, 0]]).reshape(1, 1, 3, 3)
edges = F.conv2d(img.unsqueeze(0), laplacian, padding=1).squeeze()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img.squeeze(), cmap='gray'); axes[0].set_title(f'original (label={label})')
axes[1].imshow(edges,         cmap='gray'); axes[1].set_title('Laplacian edges')
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

## 3. LeNet-5

经典结构（输入 1×32×32 灰度图）：

```
Conv2d(1→6, k=5)  -> ReLU -> MaxPool2d(2)   # 32×32 -> 28×28 -> 14×14
Conv2d(6→16, k=5) -> ReLU -> MaxPool2d(2)   # 14×14 -> 10×10 -> 5×5
Linear(16*5*5 → 120) -> ReLU
Linear(120 → 84) -> ReLU
Linear(84 → 10)
```

MNIST 是 28×28，所以输入做 padding 到 32×32（或者把第一个 Conv 改成 padding=2）。下面用 padding=2 的方案。
> **关于激活函数**：1998 年原版 LeNet-5 用的是 tanh / sigmoid。这里改用 ReLU——它在深网络上训练更快、更不容易梯度消失，是现代 CNN 的默认选择。结构其他部分保持原样。

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, n_class=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)   # 28x28 -> 28x28
        self.pool1 = nn.MaxPool2d(2, 2)                          # -> 14x14
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)             # -> 10x10
        self.pool2 = nn.MaxPool2d(2, 2)                          # -> 5x5
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84,  n_class)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# 看每层输出形状：显式 forward 一步一步走，不依赖 __init__ 属性顺序
net = LeNet5()
x = torch.zeros(1, 1, 28, 28)
print(f'input : {tuple(x.shape)}')
x = net.conv1(x); print(f'conv1 : {tuple(x.shape)}')
x = net.pool1(x); print(f'pool1 : {tuple(x.shape)}')
x = net.conv2(x); print(f'conv2 : {tuple(x.shape)}')
x = net.pool2(x); print(f'pool2 : {tuple(x.shape)}')
x = x.flatten(start_dim=1); print(f'flat  : {tuple(x.shape)}')
x = net.fc1(x);   print(f'fc1   : {tuple(x.shape)}')
x = net.fc2(x);   print(f'fc2   : {tuple(x.shape)}')
x = net.fc3(x);   print(f'fc3   : {tuple(x.shape)}')

### 参数量

In [ ]:
total = sum(p.numel() for p in LeNet5().parameters())
print(f'total params: {total:,}')
for name, p in LeNet5().named_parameters():
    print(f'  {name:18s} {str(tuple(p.shape)):20s} {p.numel():>8,d}')

## 4. 在 MNIST 子集上训练

为了 notebook 跑得快，只用 5000 训练 + 1000 测试。完整 60000/10000 在更大 epoch 下能轻松到 99% 以上。

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_full = datasets.MNIST(CACHE, train=True,  download=True, transform=transform)
test_full  = datasets.MNIST(CACHE, train=False, download=True, transform=transform)

torch.manual_seed(0)
tr_idx = torch.randperm(len(train_full))[:5000]
te_idx = torch.randperm(len(test_full))[:1000]
train_set = Subset(train_full, tr_idx.tolist())
test_set  = Subset(test_full,  te_idx.tolist())
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=128)
print(f'train {len(train_set)}  test {len(test_set)}')

In [ ]:
torch.manual_seed(0)
model = LeNet5()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    running, n = 0.0, 0
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item() * x.size(0); n += x.size(0)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item(); total += y.size(0)
    print(f'epoch {epoch+1}: train_loss={running/n:.4f}  test_acc={correct/total:.4f}')

### 看几个预测

In [ ]:
model.eval()
x, y = next(iter(test_loader))
with torch.no_grad():
    probs = F.softmax(model(x[:8]), dim=1)
preds = probs.argmax(dim=1)

fig, axes = plt.subplots(1, 8, figsize=(14, 2.5))
for ax, img, true, p, pr in zip(axes, x[:8], y[:8], preds, probs):
    ax.imshow(img.squeeze(), cmap='gray')
    ok = '✓' if true == p else '✗'
    ax.set_title(f'{ok} pred={p.item()}({pr[p].item():.2f}) true={true.item()}', fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

### 看第一个 conv 层学到的 kernel

6 个 5×5 的 filter。训练前是随机噪声，训练后会形成边缘、纹理之类的探测器。

In [ ]:
kernels = model.conv1.weight.detach().squeeze(1)   # [6, 5, 5]
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, k in zip(axes, kernels):
    ax.imshow(k, cmap='RdBu_r'); ax.axis('off')
plt.suptitle('conv1 learned kernels'); plt.show()

## 小结

- 深度学习里的 "conv" 实际是 cross-correlation，`nn.Conv2d` 即是该操作。
- 形状公式：$H_\text{out} = \lfloor (H + 2p - k) / s \rfloor + 1$；用 padding=k//2 + stride=1 可以保持空间尺寸不变。
- LeNet-5 在 MNIST 5000 训练子集上 5 个 epoch 能到 ~95% 测试准确率；完整 60k 训练 + 更多 epoch 可以到 99%。
- 第一卷积层的 kernel 在训练后会变成边缘 / 纹理探测器，可直接 imshow 看。
- chap5 下做 ResNet：用残差连接训更深的网络，在 CIFAR-10 上看效果。